### Section 0: Setup & Data Loading

This cell imports the libraries we need and loads both CSVs directly from GitHub into pandas DataFrames. It prints the shapes of each dataframes. It also prints column names so we can decide which shared column to use to merge the two datasources.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CLIMATE_URL = "https://raw.githubusercontent.com/mdkamrulhasan/data_mining_kdd/refs/heads/main/data/climate-change/climate_distortion.csv"
REGION_URL  = "https://raw.githubusercontent.com/mdkamrulhasan/data_mining_kdd/refs/heads/main/data/climate-change/country_region.csv"

climate = pd.read_csv(CLIMATE_URL)
country_region = pd.read_csv(REGION_URL)

print("\nclimate_distortion.csv shape:", climate.shape)
print("\nColumns:", climate.columns.tolist())

print("\ncountry_region.csv shape:", country_region.shape)
print("\nColumns:", country_region.columns.tolist())

shared_cols = sorted(set(climate.columns) & set(country_region.columns))
print("\nShared column(s) usable as merge key:", shared_cols)

print()
climate.head()

### Section 1: Null Values & Missingness Pattern - Task a.i

This cell counts NULLs per column in each file and reports how many columns are affected. It then looks at where the NULLs fall to find the patterns. How many rows are affected, whether the two columns go missing together, and how the affected rows are spread across regions, crop types, and years.

In [ ]:
null_climate = climate.isnull().sum()
null_region = country_region.isnull().sum()

climate_null_col_count = (null_climate > 0).sum()
region_null_col_count = (null_region > 0).sum()

print("\nColumns with NULLs in climate_distortion.csv: ", climate_null_col_count)
print(null_climate[null_climate > 0])

print("\nColumns with NULLs in country_region.csv: ", region_null_col_count)
print(null_region[null_region > 0])

null_cols = null_climate[null_climate > 0].index.tolist()
null_mask = climate[null_cols].isnull()
null_rows = climate[null_mask.any(axis=1)]
null_years = pd.to_datetime(null_rows["Date"]).dt.year

null_cell_count = null_mask.values.sum()
null_cell_pct = round(null_cell_count / climate.size * 100, 3)

null_row_count = len(null_rows)
total_row_count = len(climate)
null_row_pct = round(null_row_count / total_row_count * 100, 3)

both_null_count = null_mask.all(axis=1).sum()

print("\nNULL cells: ", null_cell_count)
print("\nPercent of NULL cells (%): ", null_cell_pct)

print("\nRows with at least one NULL: ", null_row_count)
print("\nPercent of rows with at least one NULL (%): ", null_row_pct)

print("\nRows where both columns are NULL: ", both_null_count)

affected_regions = null_rows["Region"].nunique()
total_regions = climate["Region"].nunique()
affected_crops = null_rows["Crop_Type"].nunique()
total_crops = climate["Crop_Type"].nunique()
affected_years = null_years.nunique()
min_year = null_years.min()
max_year = null_years.max()

print("\nAffected rows span regions: ", affected_regions)
print("\nTotal regions: ", total_regions)
print("\nAffected rows span crop types: ", affected_crops)
print("\nTotal crop types: ", total_crops)
print("\nAffected rows span years: ", affected_years)
print("\nEarliest affected year: ", min_year)
print("\nLatest affected year: ", max_year)

most_null_region = null_rows["Region"].value_counts().max()
most_null_year = null_years.value_counts().max()

print("\nMost NULL rows in any single region: ", most_null_region)
print("\nMost NULL rows in any single year: ", most_null_year)

Only two columns contain NULL values across both files: `Soil_Health_Index` (9 missing) and `Adaptation_Strategies` (12 missing), both in _climate_distortion.csv_. No columns in _country_region.csv_ have missing values. The absence is sparse and appears to be scattered and random rather than following a systematic pattern tied to a specific country, region, or time period. This is consistent with isolated data-entry gaps.

### Section 2: Impute NULL Values - Task a.ii

Imputation is done here. Numeric NULLs are filled with the column median and categorical NULLs with the column mode (most frequent category). _country_region.csv_ has no NULLs, so it needs no imputation.


In [ ]:
climate_clean = climate.copy()

numeric_cols = climate_clean.select_dtypes(include=[np.number]).columns
categorical_cols = climate_clean.select_dtypes(exclude=[np.number]).columns

for col in numeric_cols:
    n_missing = climate_clean[col].isnull().sum()
    if n_missing:
        fill_value = climate_clean[col].median()
        climate_clean[col] = climate_clean[col].fillna(fill_value)

        mean_value = round(climate[col].mean(), 3)
        skew_value = round(climate[col].skew(), 3)
        fill_value_rounded = round(fill_value, 3)

        print("\nColumn: ", col)
        print("\nFilled NULL count: ", n_missing)
        print("\nFill value (median): ", fill_value_rounded)

print()
for col in categorical_cols:
    n_missing = climate_clean[col].isnull().sum()
    if n_missing:
        fill_value = climate_clean[col].mode()[0]
        climate_clean[col] = climate_clean[col].fillna(fill_value)

        print("\nColumn: ", col)
        print("\nFilled NULL count: ", n_missing)
        print("\nFill value (mode): ", fill_value)

print()
remaining_nulls = climate_clean.isnull().sum().sum()
print("\nRemaining NULLs after imputation: ", remaining_nulls)

`Soil_Health_Index` (numeric) was imputed with the column **median**, and `Adaptation_Strategies` (categorical) was imputed with the column **mode** (most frequent category).

The median was chosen over the mean for `Soil_Health_Index` because it is robust to outliers and skew, so it does not distort the variable's center.

The mode was chosen for `Adaptation_Strategies` because for a categorical field, the most frequently observed strategy is the most statistically likely value to have occurred in place of a missing entry.

Both methods allow us to retain more data than simply dropping rows with missing values.

### Section 3: Unique Regions for USA - Task a.iii

_climate_distortion.csv_ records only `Region`, so the country for each row comes from _country_region.csv_. This cell left-joins the imputed climate data (`climate_clean`) with `country_region` on `Region`. The printed row count is unchanged and confirms that every region found a country. The cell then filters to USA rows and counts the distinct region values recorded for the USA.


In [ ]:
climate = climate_clean.merge(country_region, on="Region", how="left")

print("\nRows before merge:", len(climate_clean), "| after merge:", len(climate))
print("\nUnmatched Region rows (no Country found):", climate["Country"].isnull().sum())
print("\nColumns after merge:", climate.columns.tolist())

usa_mask = climate["Country"].astype(str).str.contains("USA", case=False, na=False)
usa_data = climate[usa_mask].copy()

print("\nUnique Region(s) for USA:", usa_data["Region"].nunique())
print(usa_data["Region"].unique())

### Section 4: Descriptive Statistics - Task b
For USA data only, this cell computes mean, median, variance, standard deviation, min, max, Q1, Q3, and IQR for CO<sub>2</sub> Emissions MT and Pesticide Use KG per HA.

In [ ]:
def describe_var(series, name):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    mean = series.mean()
    std = series.std()
    cv = (std / mean) * 100

    stats = {
        "Mean": mean,
        "Median": series.median(),
        "Variance": series.var(),
        "Std Dev": std,
        "CV (%)": cv,
        "Min": series.min(),
        "Max": series.max(),
        "Q1": q1,
        "Q3": q3,
        "IQR": q3 - q1,
    }

    print()
    print("---", name, "---")
    for k, v in stats.items():
        print(k, ":", round(v, 3))
    print()

    return stats


co2_stats = describe_var(
    usa_data["CO2_Emissions_MT"],
    "CO2 Emissions MT (USA)"
)

pest_stats = describe_var(
    usa_data["Pesticide_Use_KG_per_HA"],
    "Pesticide Use KG per HA (USA)"
)

Both variables show similar relative variability, with coefficients of variation of approximately 58.6% for Pesticide Use and 57.4% for CO<sub>2</sub> Emissions. This means their values vary by a similar amount relative to their respective averages.

Additionally, the mean and median are very close for both variables, suggesting that neither distribution is strongly skewed.

### Section 5: CO<sub>2</sub> Extremes Before/After 2000 - Task c.i

This cell sums the recorded CO<sub>2</sub> Emissions MT values for each country till 2000 (inclusive) and after 2000 separately. It reports the countries with the highest and lowest aggregate values in each period.


In [ ]:
climate["Year"] = pd.to_datetime(climate["Date"]).dt.year
before_2000 = climate[climate["Year"] <= 2000]
after_2000 = climate[climate["Year"] > 2000]


def extremes(df, label):
    country_totals = df.groupby("Country")["CO2_Emissions_MT"].sum()

    max_country = country_totals.idxmax()
    min_country = country_totals.idxmin()

    print(
        label, "— Highest aggregate CO2:",
        max_country, "(", round(country_totals[max_country], 3), "MT)"
    )
    print(
        label, "— Lowest aggregate CO2:",
        min_country, "(", round(country_totals[min_country], 3), "MT)"
    )

extremes(before_2000, "Till 2000")
print()
extremes(after_2000, "After 2000")

### Section 6: Coolest Date in USA - Task c.ii

This cell finds the row with the minimum Average Temperature C among USA records and reports its date.

In [ ]:
coolest_row = usa_data.loc[usa_data["Average_Temperature_C"].idxmin()]

coolest_date = coolest_row["Date"]
coolest_temp = round(coolest_row["Average_Temperature_C"], 3)

print("Coolest date in USA: ", coolest_date, "(Avg Temp =", coolest_temp,"C)")

### Section 7: Highest-Impact Crop Type - Task c.iii

This cell filters to the Midwest(U) and Northeast(U) regions, sums Economic Impact Million USD by Crop Type, and reports which crop had the highest combined impact.

In [ ]:
regions_of_interest = ["Midwest(U)", "Northeast(U)"]
subset = climate[climate["Region"].isin(regions_of_interest)]
crop_impact = subset.groupby("Crop_Type")["Economic_Impact_Million_USD"].sum().sort_values(ascending=False)

print("\nEconomic Impact by Crop Type (Midwest(U) + Northeast(U)):")
print(crop_impact)

top_crop = crop_impact.idxmax()
top_crop_impact = round(crop_impact.max(), 3)

print("\nHighest-impact crop: ", top_crop, " (", top_crop_impact, " Million USD)")

### Section 8: Strongest/Weakest Association - Task c.iv

This cell computes the correlation matrix across the 8 listed variables and identifies the pair with the highest and lowest absolute correlation.

In [ ]:
vars8 = ["Average_Temperature_C", "Total_Precipitation_mm", "CO2_Emissions_MT",
         "Extreme_Weather_Events", "Pesticide_Use_KG_per_HA", "Fertilizer_Use_KG_per_HA",
         "Soil_Health_Index", "Economic_Impact_Million_USD"]

corr = climate[vars8].corr()
corr_abs = corr.abs().copy()
np.fill_diagonal(corr_abs.values, np.nan)

max_pair = corr_abs.stack().idxmax()
min_pair = corr_abs.stack().idxmin()

max_r = round(corr.loc[max_pair], 3)
min_r = round(corr.loc[min_pair], 3)

print("\nHighest association: ", max_pair, " -> r = ", max_r)
print("\nLowest  association: ", min_pair, " -> r = ", min_r)

The strongest association among the eight variables was between `Average_Temperature_C` and `Economic_Impact_Million_USD` (_r = 0.196_).

The weakest was between `Fertilizer_Use_KG_per_HA` and `Soil_Health_Index` (_r = -0.0_, essentially no relationship).

This suggests that as temperature increases, there is a tendency for economic impact to increase as well. The near-zero correlation between `Fertilizer_Use_KG_per_HA` and `Soil_Health_Index` suggests these two variables vary independently of one another in this sample. Fertilizer application levels show no linear tendency to move together with soil health scores.

### Section 9: Histograms - Task d.i

This creates side-by-side histograms for CO<sub>2</sub> Emissions MT and Pesticide Use KG per HA (USA only).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(usa_data["CO2_Emissions_MT"].dropna(), bins=20, color="steelblue", edgecolor="black")
axes[0].set_title("Histogram: CO2 Emissions MT (USA)")
axes[0].set_xlabel("CO2 Emissions MT"); axes[0].set_ylabel("Frequency")

axes[1].hist(usa_data["Pesticide_Use_KG_per_HA"].dropna(), bins=20, color="darkorange", edgecolor="black")
axes[1].set_title("Histogram: Pesticide Use KG per HA (USA)")
axes[1].set_xlabel("Pesticide Use KG per HA"); axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("co2_pesticide_histograms.png", dpi=150)
plt.show()

Both CO<sub>2</sub> Emissions MT and Pesticide Use KG per HA in the USA histograms appear approximately symmetric and closely resemble a uniform distribution rather than a normal, skewed, or bimodal one.

### Section 10: Yearly Precipitation Trend - Task d.ii

This cell plots the yearly average Total Precipitation mm for the USA as a line chart to describe any visible trend.

In [ ]:
usa_data["Year"] = pd.to_datetime(usa_data["Date"]).dt.year
yearly_precip = usa_data.groupby("Year")["Total_Precipitation_mm"].mean()

plt.figure(figsize=(10, 5))
plt.plot(yearly_precip.index, yearly_precip.values, marker="o", color="teal")
plt.title("Yearly Average Total Precipitation (USA)")
plt.xlabel("Year"); plt.ylabel("Average Total Precipitation (mm)")
plt.grid(True, alpha=0.3)
plt.savefig("yearly_precipitation_trend.png", dpi=150)
plt.show()

The yearly average `Total_Precipitation_mm` for the USA does not show a clear long-term upward or downward trend across the years. The line fluctuates from year to year without a consistent directional pattern. Any single-year peaks or dips should be read as sampling variability rather than evidence of a genuine precipitation trend.

### Section 11: Scatter Plot for Chosen Pair - Task d.iii

This cell plots the two variables from Section 8 with the highest correlation to support the interpretation of that association.

In [ ]:
var_x, var_y = max_pair

axis_labels = {
    "Average_Temperature_C": "Average temperature (°C)",
    "Total_Precipitation_mm": "Total precipitation (mm)",
    "CO2_Emissions_MT": "CO2 emissions (MT)",
    "Extreme_Weather_Events": "Extreme weather events (count)",
    "Pesticide_Use_KG_per_HA": "Pesticide use (kg/ha)",
    "Fertilizer_Use_KG_per_HA": "Fertilizer use (kg/ha)",
    "Soil_Health_Index": "Soil health index (unitless)",
    "Economic_Impact_Million_USD": "Economic impact (million USD)",
}

plt.figure(figsize=(7, 6))
plt.scatter(climate[var_x], climate[var_y], alpha=0.4, color="purple")
plt.title(
    f"Scatter Plot: {var_x} vs {var_y} "
    f"(r = {corr.loc[max_pair]:.3f})"
)
plt.xlabel(axis_labels[var_x])
plt.ylabel(axis_labels[var_y])
plt.tight_layout()
plt.savefig("scatter_highest_association.png", dpi=150)
plt.show()

The scatter plot of `Average_Temperature_C` vs. `Economic_Impact_Million_USD` shows a wide, diffuse cloud of points with only a slight upward drift. This is consistent with the positive correlation identified in Section 8 (r = 0.196), but the correlation is weak.

This visualization supports the interpretation that while there may be a marginal tendency for economic impact to rise with temperature, the relationship is far from linear or deterministic. Points are scattered broadly across the full range of both variables with no tight clustering along a trend line. This reinforces that temperature by itself is a weak predictor of economic impact in this sample, and other factors not captured by this single pair likely drive most of the variation.